# Analysing Teams' Performance - Batch Processing - Silver Layer



- Aggregating goals scored by each team and Total goals per team
- Aggregating substitutions made by each team and Total substitutions per team
- Aggregating bookings (Yellow/Red cards) received by each team and Total bookings per team
- Aggregating penalty kicks taken and converted by each team and Total penalty per team

In [0]:
# And uninstall a few helpers we prepared for you
%pip uninstall -y databricks_helpers exercise_ev_databricks_unit_tests

# now install the databricks helpers 1
%pip install git+https://github.com/data-derp/databricks_helpers.git

# # now install and also the databricks test cases
%pip install git+https://github.com/data-derp/exercise_ev_databricks_unit_tests.git

In [0]:
from databricks_helpers.databricks_helpers import DataDerpDatabricksHelpers
exercise_name = "team_analysis_batch_processing/silver"
helpers = DataDerpDatabricksHelpers(dbutils, exercise_name)

current_user = helpers.current_user()
working_directory = helpers.working_directory()
print(f"Your current working directory is: {working_directory}")

## This function CLEARS your current working directory. Only run this if you want a fresh start or if it is the first time you're doing this exercise.
helpers.clean_working_directory()
bronze_working_directory = working_directory.replace("silver", "bronze")


In [0]:
bronze_output_teams = f"{bronze_working_directory}/output/teams"
bronze_output_goals = f"{bronze_working_directory}/output/goals"
bronze_output_bookings = f"{bronze_working_directory}/output/bookings"
bronze_output_penalty_kicks = f"{bronze_working_directory}/output/penalty_kicks"
bronze_output_substitutions = f"{bronze_working_directory}/output/substitutions"
bronze_output_squads = f"{bronze_working_directory}/output/squads"

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql import functions as F

def read_parquet(filepath: str) -> DataFrame:
    df = spark.read.parquet(filepath)
    return df
    
teamss_bronze_df = read_parquet(bronze_output_teams)
goals_bronze_df=read_parquet(bronze_output_goals)
bookings_bronze_df=read_parquet(bronze_output_bookings)
penalty_kicks_bronze_df=read_parquet(bronze_output_penalty_kicks)
substitutions_bronze_df=read_parquet(bronze_output_substitutions)
squads_bronze_df = read_parquet(bronze_output_squads)

In [0]:
goals_bronze_df=goals_bronze_df.withColumn("year", F.year("match_date"))
bookings_bronze_df=bookings_bronze_df.withColumn("year", F.year("match_date"))
penalty_kicks_bronze_df=penalty_kicks_bronze_df.withColumn("year", F.year("match_date"))
substitutions_bronze_df=substitutions_bronze_df.withColumn("year", F.year("match_date"))

# Filter only 2022 sets
# goals_bronze_df = goals_bronze_df.filter(
#     F.col("year").between(1950, 2022)
# )
goals_bronze_df=goals_bronze_df
bookings_bronze_df=bookings_bronze_df
penalty_kicks_bronze_df=penalty_kicks_bronze_df
substitutions_bronze_df=substitutions_bronze_df

display(goals_bronze_df)
display(substitutions_bronze_df)

### **Goals Aggregation (Team Level)**

In [0]:
goals_per_team = goals_bronze_df.select(
    "match_id",
    "team_id",
    "team_code",
    "team_name",
    "match_date",
    "minute_regulation",
    "own_goal",
    "penalty",
    "stage_name"
).withColumn(
    "year", F.year("match_date")
).withColumn(
    "is_early_goal",
    F.when(F.col("minute_regulation") <= 15, 1).otherwise(0)
)
display(goals_per_team)

In [0]:
def write(input_df: DataFrame):
    out_dir = f"{working_directory}/output/goals_per_team"
    
### Put your code here.
    mode_name = "overwrite"
    
    input_df. \
        write. \
        mode(mode_name). \
        parquet(out_dir)
    
    
write(goals_per_team)
dbutils.fs.ls(f"{working_directory}/output/goals_per_team")

### **Booking aggregations**

In [0]:
bookings_by_team = bookings_bronze_df.select(
    "match_id",
    "team_id",
    "team_code",
    "team_name",
    "match_date",
    "minute_regulation",
    "yellow_card",
    "red_card",
    "second_yellow_card",
    "sending_off"
).withColumn(
    "year", F.year("match_date")
).withColumn(
    "is_yellow_card",
    F.when(F.col("yellow_card") == 1, 1).otherwise(0)
).withColumn(
    "is_red_card",
    F.when(
        (F.col("red_card") == 1) | 
        (F.col("second_yellow_card") == 1), 1
    ).otherwise(0)
).withColumn(
    "is_direct_red",
    F.when(F.col("red_card") == 1, 1).otherwise(0)
).withColumn(
    "is_indirect_red",
    F.when(F.col("second_yellow_card") == 1, 1).otherwise(0)
).withColumn(
    "is_sending_off",
    F.when(F.col("sending_off") == 1, 1).otherwise(0)
).withColumn(
    "is_early_booking",
    F.when(F.col("minute_regulation") <= 30, 1).otherwise(0)
).withColumn(
    "is_late_booking",
    F.when(F.col("minute_regulation") >= 75, 1).otherwise(0)
)
display(bookings_by_team)

In [0]:
def write(input_df: DataFrame):
    out_dir = f"{working_directory}/output/bookings_by_team"
    
### Put your code here.
    mode_name = "overwrite"
    
    input_df. \
        write. \
        mode(mode_name). \
        parquet(out_dir)
    
    
write(bookings_by_team)
dbutils.fs.ls(f"{working_directory}/output/bookings_by_team")

### **Substitutions Aggregation**

In [0]:
substitutions_by_team = substitutions_bronze_df.select(
    "match_id",
    "team_id",
    "team_code",
    "team_name",
    "match_date",
    "minute_regulation",
    "substitution_type"   # "in" / "out"
).withColumn(
    "year", F.year("match_date")
).withColumn(
    "is_early_sub",
    F.when(F.col("minute_regulation") <= 30, 1).otherwise(0)
).withColumn(
    "is_late_sub",
    F.when(F.col("minute_regulation") >= 75, 1).otherwise(0)
).withColumn(
    "is_player_in",
    F.when(F.col("substitution_type") == "in", 1).otherwise(0)
).withColumn(
    "is_player_out",
    F.when(F.col("substitution_type") == "out", 1).otherwise(0)
)
display(substitutions_by_team)

In [0]:
def write(input_df: DataFrame):
    out_dir = f"{working_directory}/output/substitutions_by_team"
    
### Put your code here.
    mode_name = "overwrite"
    
    input_df. \
        write. \
        mode(mode_name). \
        parquet(out_dir)
    
    
write(substitutions_by_team)
dbutils.fs.ls(f"{working_directory}/output/substitutions_by_team")

### **Penalty Aggregations**

In [0]:
penalty_kicks_by_team = penalty_kicks_bronze_df.select(
    "match_id",
    "team_id",
    "team_code",
    "team_name",
    "match_date",
    "tournament_name",
    "stage_name",
    "converted"
).withColumn(
    "year", F.year("match_date")
).withColumn(
    # Clean conversion
    "converted_clean",
    F.when(F.col("converted") == 1, 1)
     .when(F.col("converted") == 0, 0)
     .otherwise(0)
).withColumn(
    "is_penalty_scored",
    F.col("converted_clean")
).withColumn(
    "is_penalty_missed",
    F.when(F.col("converted_clean") == 0, 1).otherwise(0)
)

display(penalty_kicks_by_team)

In [0]:
def write(input_df: DataFrame):
    out_dir = f"{working_directory}/output/penalty_kicks_by_team"
    
### Put your code here.
    mode_name = "overwrite"
    
    input_df. \
        write. \
        mode(mode_name). \
        parquet(out_dir)
    
    
write(penalty_kicks_by_team)
dbutils.fs.ls(f"{working_directory}/output/penalty_kicks_by_team")